Team coursework: CatBoost classification on an anonymised Kaggle-style tabular dataset (class-imbalanced binary target), part of the same assignment as `svm_imbalance_and_tuning.ipynb` in this folder. The input CSV and rendered outputs are not included, and results are not independently reproduced in this portfolio copy.


# START

In [ ]:
# ==========================================
# LIBRARY IMPORTS
# ==========================================

print("="*80)
print("IMPORTING LIBRARIES")
print("="*80)

# Core data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn - Model selection and metrics
from sklearn.model_selection import (
    train_test_split, 
    GridSearchCV, 
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
    matthews_corrcoef
)

# Scikit-learn - Preprocessing
from sklearn.impute import SimpleImputer

# Scikit-learn - Feature selection
from sklearn.feature_selection import SelectFromModel

# Imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import EasyEnsembleClassifier

# CatBoost
from catboost import CatBoostClassifier

# Optuna
import optuna

# Scipy
import scipy  # ✅ Import scipy directly
from scipy.stats import randint, uniform
from scipy import stats

# Time tracking
import time

# Warnings (optional - to suppress warnings)
import warnings
warnings.filterwarnings('ignore')

print("\n✅ All libraries imported successfully!")

# ==========================================
# VERIFY LIBRARY VERSIONS
# ==========================================

print("\n" + "="*80)
print("LIBRARY VERSIONS")
print("="*80)

import sklearn
import imblearn
import catboost

print(f"✅ pandas version: {pd.__version__}")
print(f"✅ numpy version: {np.__version__}")
print(f"✅ matplotlib version: {plt.matplotlib.__version__}")
print(f"✅ seaborn version: {sns.__version__}")
print(f"✅ scikit-learn version: {sklearn.__version__}")
print(f"✅ imbalanced-learn version: {imblearn.__version__}")
print(f"✅ catboost version: {catboost.__version__}")
print(f"✅ optuna version: {optuna.__version__}")
print(f"✅ scipy version: {scipy.__version__}")  # ✅ Fixed!

print("\n" + "="*80)
print("ALL LIBRARIES READY!")
print("="*80)

원래대로 되돌리고 싶으면
pd.reset_option('display.max_rows')

## Data Understanding

In [ ]:
df_eda = pd.read_csv("datafile_full.csv")

In [ ]:
# ==========================================
# BASIC DATA EXPLORATION
# ==========================================
print("\n" + "-"*80)
print("BASIC DATA EXPLORATION")
print("-"*80)

print(df_eda.shape)
#it shows the number of rows and columns
#it has 20000 rows and 202 columns

In [ ]:
df_eda.head()
#For the target column, the value = 1 if the customer applied for business credit and 0 otherwise.

In [ ]:
df_eda.describe()
# data is masked

In [ ]:
# Check the structure (data types and non-null counts) of the variables in the dataset by using info() method
#data types: 201 float64 and 1 int64 (all numerical, therefore no need to change data type)
df_eda.info()

In [ ]:
df_eda.dtypes[df_eda.dtypes == 'int64']

# target balance check

In [ ]:
#it indicates our target variable is highly imbalanced.
df_eda['target'].value_counts()

In [ ]:
#df_full['target'].value_counts(normalize=True)
print(f"Target distribution:\n{df['target'].value_counts(normalize=True)}")

In [ ]:
df_eda['target'].value_counts().plot(kind='bar')
plt.title('Target Variable Distribution')
plt.xlabel('Target')
plt.ylabel('Count')
plt.show()


# Duplicate row check

In [ ]:
#Check for duplicates
print(f"\nDuplicate rows: {df_eda.duplicated().sum()}")

In [ ]:
#Check ID_code (should be unique)
print(f"Unique ID_codes: {df_eda['ID_code'].nunique()}")
print(f"Is ID_code unique? {df_eda['ID_code'].nunique() == len(df_eda)}")

# Missing-Values Analysis

In [ ]:
# Overall missing values
total_missing = df_eda.isnull().sum().sum()
total_cells = df_eda.size
overall_pct = (total_missing / total_cells) * 100

print(f"\nOverall Missing Value Summary:")
print(f"  Total missing values: {total_missing:,}")
print(f"  Total cells: {total_cells:,}")
print(f"  Overall missing percentage: {overall_pct:.2f}%")


CHECK ID_CODE AND TARGET FOR MISSING VALUES

In [ ]:
# Check ID_code
id_missing = df_eda['ID_code'].isnull().sum()
id_missing_pct = (id_missing / len(df_eda)) * 100
print(f"\nID_code:")
print(f"  Missing values: {id_missing}")
print(f"  Missing percentage: {id_missing_pct:.2f}%")
if id_missing == 0:
    print(f"  ✅ No missing values in ID_code")
else:
    print(f"  ⚠️  ID_code has missing values!")

# Check target
target_missing = df_eda['target'].isnull().sum()
target_missing_pct = (target_missing / len(df_eda)) * 100
print(f"\ntarget:")
print(f"  Missing values: {target_missing}")
print(f"  Missing percentage: {target_missing_pct:.2f}%")
if target_missing == 0:
    print(f"  ✅ No missing values in target")
else:
    print(f"  ⚠️  target has missing values!")

In [ ]:
# Missing values per column
missing_per_column = df_eda.isnull().sum()
missing_pct_per_column = (missing_per_column / len(df_eda)) * 100

# Create summary dataframe
missing_summary = pd.DataFrame({
    'Column': missing_per_column.index,
    'Missing_Count': missing_per_column.values,
    'Missing_Percentage': missing_pct_per_column.values
}).sort_values('Missing_Count', ascending=False)

# Only columns with missing values
columns_with_missing = missing_summary[missing_summary['Missing_Count'] > 0]

if len(columns_with_missing) > 0:
    print(f"\n✅ Found {len(columns_with_missing)} columns with missing values:")
    print("\nTop 20 columns with most missing values:")
    print(columns_with_missing.head(20).to_string(index=False))
else:
    print("\n✅ No missing values found in any column")

MISSING VALUE DISTRIBUTION BY CATEGORY

In [ ]:
# MISSING VALUE DISTRIBUTION BY CATEGORY
print("\n" + "-"*80)
print("MISSING VALUE DISTRIBUTION BY CATEGORY")
print("-"*80)

# Get feature columns only (exclude ID_code and target)
feature_cols = [col for col in df_eda.columns if col.startswith('var_')]
feature_missing_summary = missing_summary[missing_summary['Column'].isin(feature_cols)]

print(f"\nFeature columns only (var_*):")
print(f"  Columns with 0% missing: {(feature_missing_summary['Missing_Percentage'] == 0).sum()}")
print(f"  Columns with 1-5% missing: {((feature_missing_summary['Missing_Percentage'] > 0) & (feature_missing_summary['Missing_Percentage'] <= 5)).sum()}")
print(f"  Columns with 5-10% missing: {((feature_missing_summary['Missing_Percentage'] > 5) & (feature_missing_summary['Missing_Percentage'] <= 10)).sum()}")
print(f"  Columns with >10% missing: {(feature_missing_summary['Missing_Percentage'] > 10).sum()}")

print(f"\nAll columns (including ID_code and target):")
print(f"  Columns with 0% missing: {(missing_summary['Missing_Percentage'] == 0).sum()}")
print(f"  Columns with 1-5% missing: {((missing_summary['Missing_Percentage'] > 0) & (missing_summary['Missing_Percentage'] <= 5)).sum()}")
print(f"  Columns with 5-10% missing: {((missing_summary['Missing_Percentage'] > 5) & (missing_summary['Missing_Percentage'] <= 10)).sum()}")
print(f"  Columns with >10% missing: {(missing_summary['Missing_Percentage'] > 10).sum()}")


IDENTIFY HIGH MISSING VALUE COLUMNS

In [ ]:
# IDENTIFY HIGH MISSING VALUE COLUMNS
print("\n" + "-"*80)
print("HIGH MISSING VALUE COLUMNS (>30%)")
print("-"*80)

high_missing = columns_with_missing[columns_with_missing['Missing_Percentage'] > 30]

if len(high_missing) > 0:
    print(f"\n⚠️  Found {len(high_missing)} columns with >30% missing values:")
    print(high_missing.to_string(index=False))
    print("\n💡 Recommendation: Consider dropping these columns")
else:
    print("\n✅ No columns with >30% missing values")

We have 200 features.  
there is no missing values.  
data types are all numerical  (target = int64(1), the others = float64(201)).  
90% - target 0 , 10% - target - 1  
Class Imbalance: 90.18% class 0, 9.82% class 1 (SEVERE IMBALANCE!)  
* Null-Values 
Columns with 0% missing: 131  
Columns with 1-5% missing: 68  
Columns with 5-10% missing: 2  
Columns with >10% missing: 1  

# Outliers

In [ ]:
# Define function to count outliers using IQR method
def count_outliers(df, column):
    """
    Count outliers using IQR (Interquartile Range) method
    Outliers are defined as values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers)

# Get all feature columns
feature_cols = [col for col in df_eda.columns if col.startswith('var_')]

print(f"\nAnalyzing outliers for {len(feature_cols)} features using IQR method...")

# Calculate outliers for all features
outlier_data = []

for feature in feature_cols:
    outlier_count = count_outliers(df_eda, feature)
    outlier_pct = (outlier_count / len(df_eda)) * 100
    
    outlier_data.append({
        'Feature': feature,
        'Outlier_Count': outlier_count,
        'Outlier_Percentage': outlier_pct
    })

# Create DataFrame sorted by outlier count
outlier_df = pd.DataFrame(outlier_data).sort_values(
    'Outlier_Count', 
    ascending=False
)

# OUTLIER SUMMARY STATISTICS
print("\n" + "-"*80)
print("OUTLIER SUMMARY")
print("-"*80)

total_outliers = outlier_df['Outlier_Count'].sum()
avg_outliers = outlier_df['Outlier_Count'].mean()
median_outliers = outlier_df['Outlier_Count'].median()
features_with_high_outliers = (outlier_df['Outlier_Percentage'] > 10).sum()

print(f"\nTotal features analyzed: {len(outlier_df)}")
print(f"Total outliers across all features: {total_outliers:,}")
print(f"Average outliers per feature: {avg_outliers:.1f}")
print(f"Median outliers per feature: {median_outliers:.1f}")
print(f"Features with >10% outliers: {features_with_high_outliers}")

# OUTLIER DISTRIBUTION BY CATEGORY
print("\n" + "-"*80)
print("FEATURES BY OUTLIER PERCENTAGE")
print("-"*80)

high_outlier_features = (outlier_df['Outlier_Percentage'] > 10).sum()
medium_outlier_features = ((outlier_df['Outlier_Percentage'] > 5) & 
                           (outlier_df['Outlier_Percentage'] <= 10)).sum()
low_outlier_features = (outlier_df['Outlier_Percentage'] <= 5).sum()

print(f"\n  >10% outliers: {high_outlier_features} features ({high_outlier_features/len(outlier_df)*100:.1f}%)")
print(f"  5-10% outliers: {medium_outlier_features} features ({medium_outlier_features/len(outlier_df)*100:.1f}%)")
print(f"  ≤5% outliers: {low_outlier_features} features ({low_outlier_features/len(outlier_df)*100:.1f}%)")


In [ ]:
# OUTLIER VISUALIZATIONS
print("\n" + "-"*80)
print("CREATING OUTLIER VISUALIZATIONS")
print("-"*80)

# Set global style
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of outlier percentages
axes[0].hist(outlier_df['Outlier_Percentage'], bins=30, 
             color='gray', edgecolor='black', alpha=0.7)
axes[0].axvline(outlier_df['Outlier_Percentage'].mean(), 
                color='black', linestyle='--', linewidth=2,
                label=f'Mean: {outlier_df["Outlier_Percentage"].mean():.2f}%')
axes[0].set_xlabel('Outlier Percentage (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Features', fontsize=12, fontweight='bold')
axes[0].set_title('Distribution of Outlier Percentages Across Features', 
                  fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# Bar chart of outlier categories
categories = ['High\n(>10%)', 'Medium\n(5-10%)', 'Low\n(≤5%)']
counts = [high_outlier_features, medium_outlier_features, low_outlier_features]
colors = ['#2C2C2C', '#6E6E6E', '#B8B8B8']
bars = axes[1].bar(categories, counts, color=colors, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Number of Features', fontsize=12, fontweight='bold')
axes[1].set_title('Features by Outlier Category', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

# Add extra space at top to prevent text overlap
axes[1].set_ylim(0, max(counts) * 1.15)

# Add value labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{count}\n({count/len(outlier_df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('outlier_visualizations_grayscale.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Top 20 features with most outliers
fig, ax = plt.subplots(figsize=(12, 10))
top_20_outliers = outlier_df.head(20)
bars = ax.barh(range(len(top_20_outliers)), 
               top_20_outliers['Outlier_Percentage'], 
               color='#FF6B6B')
ax.set_yticks(range(len(top_20_outliers)))
ax.set_yticklabels(top_20_outliers['Feature'], fontsize=10)
ax.set_xlabel('Outlier Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('Top 20 Features with Highest Outlier Percentage', 
             fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('top_20_outliers.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Outlier analysis complete!")

# KEY INSIGHTS
print("\n" + "="*80)
print("KEY INSIGHTS - OUTLIERS")
print("="*80)

print(f"\n📊 Outlier Summary:")
print(f"   • {total_outliers:,} total outliers detected across all features")
print(f"   • Average of {avg_outliers:.1f} outliers per feature")
print(f"   • {high_outlier_features} features have >10% outliers")

if high_outlier_features > 0:
    print(f"\n⚠️  High outlier features detected!")
    print(f"   Consider investigating these features further")
else:
    print(f"\n✅ No features with critically high outlier percentages")

print("\n💡 Note: Outliers detected using IQR method (Q1 - 1.5*IQR, Q3 + 1.5*IQR)")
print("   Decision on handling outliers should be made during preprocessing")


# Correlation Analysis (Target vs Variables)

|Strength    | positive   | Negative     | <br>
|Very Strong | 0.7 to 1.0 | -1.0 to -0.7 | <br>
|Strong      | 0.5 to 0.7 | -0.7 to -0.5 | <br>
|Moderate    | 0.3 to 0.5 | -0.5 to -0.3 | <br>
|Weak        | 0.0 to 0.3 | -0.3 to 0.0  | <br>


In [ ]:
# ==========================================
# CORRELATION ANALYSIS WITH TARGET
# ==========================================
print("\n" + "="*80)
print("CORRELATION ANALYSIS WITH TARGET")
print("="*80)

# Get feature columns
feature_cols = [col for col in df_eda.columns if col.startswith('var_')]

print(f"\n📊 Analyzing correlations for {len(feature_cols)} features with target...\n")

# PEARSON CORRELATION
print("="*80)
print("PEARSON CORRELATION WITH TARGET")
print("="*80)

# Calculate Pearson correlations
pearson_correlations = df_eda[feature_cols].corrwith(df_eda['target'])
pos_corr = pearson_correlations[pearson_correlations > 0].sort_values(ascending=False)
neg_corr = pearson_correlations[pearson_correlations < 0].sort_values(ascending=True)

print("\n📈 Top 20 POSITIVELY Correlated Features:")
print(pos_corr.head(20))

print("\n📉 Top 20 NEGATIVELY Correlated Features:")
print(neg_corr.head(20))

# Statistics
print(f"\n📊 Correlation Statistics:")
print(f"   Maximum correlation: {pearson_correlations.max():.4f} ({pearson_correlations.idxmax()})")
print(f"   Minimum correlation: {pearson_correlations.min():.4f} ({pearson_correlations.idxmin()})")
print(f"   Mean absolute correlation: {pearson_correlations.abs().mean():.4f}")
print(f"   Median absolute correlation: {pearson_correlations.abs().median():.4f}")

# Categorize by correlation strength
strong_count = (pearson_correlations.abs() >= 0.7).sum()
moderate_count = ((pearson_correlations.abs() >= 0.3) & (pearson_correlations.abs() < 0.7)).sum()
weak_count = ((pearson_correlations.abs() > 0.0) & (pearson_correlations.abs() < 0.3)).sum()

print(f"\n📊 Features by Correlation Strength:")
print(f"   |r| ≥ 0.7 (Strong): {strong_count} features")
print(f"   0.3 ≤ |r| < 0.7 (Moderate): {moderate_count} features")
print(f"   0 < |r| < 0.3 (Weak): {weak_count} features")

# Save correlation results to DataFrame
correlation_df = pd.DataFrame({
    'Feature': pearson_correlations.index,
    'Correlation': pearson_correlations.values,
    'Abs_Correlation': pearson_correlations.abs().values
}).sort_values('Abs_Correlation', ascending=False)

print("\n📊 Top 10 Features by Absolute Correlation:")
print(correlation_df.head(10).to_string(index=False))


In [ ]:
# Visualize Top 20 Correlations
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), constrained_layout=True)

# Top 20 positive
top_20_pos = pos_corr.head(20)
ax1.barh(range(len(top_20_pos)), top_20_pos.values, color='#6E6E6E')
ax1.set_yticks(range(len(top_20_pos)))
ax1.set_yticklabels(top_20_pos.index, fontsize=10)
ax1.set_xlabel('Correlation with Target', fontsize=12, fontweight='bold')
ax1.set_title('Top 20 Positively Correlated Features', fontsize=14, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_20_pos.values):
    ax1.text(v - 0.002, i, f'{v:.4f}', va='center', ha='right', 
             fontsize=9, color='white', fontweight='bold')

# Top 20 negative
top_20_neg = neg_corr.head(20)
ax2.barh(range(len(top_20_neg)), top_20_neg.values, color='#6E6E6E')
ax2.set_yticks(range(len(top_20_neg)))
ax2.set_yticklabels(top_20_neg.index, fontsize=10)
ax2.set_xlabel('Correlation with Target', fontsize=12, fontweight='bold')
ax2.set_title('Top 20 Negatively Correlated Features', fontsize=14, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

for i, v in enumerate(top_20_neg.values):
    ax2.text(v + 0.002, i, f'{v:.4f}', va='center', ha='left', 
             fontsize=9, color='white', fontweight='bold')

plt.savefig('top_20_correlations_grayscale.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Set global style
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Plot 1: Top 20 Correlations - VALUE LABELS INSIDE BARS
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), constrained_layout=True)

# Top 20 positive
top_20_pos = pos_corr.head(20)
ax1.barh(range(len(top_20_pos)), top_20_pos.values, color='#6E6E6E')  # Changed to gray
ax1.set_yticks(range(len(top_20_pos)))
ax1.set_yticklabels(top_20_pos.index, fontsize=10)
ax1.set_xlabel('Correlation with Target', fontsize=12, fontweight='bold')
ax1.set_title('Top 20 Positively Correlated Features', fontsize=14, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# ✅ Value labels INSIDE bars (right-aligned)
for i, v in enumerate(top_20_pos.values):
    ax1.text(v - 0.002, i, f'{v:.4f}', va='center', ha='right', 
             fontsize=9, color='white', fontweight='bold')

# Top 20 negative
top_20_neg = neg_corr.head(20)
ax2.barh(range(len(top_20_neg)), top_20_neg.values, color='#6E6E6E')  # Changed to gray
ax2.set_yticks(range(len(top_20_neg)))
ax2.set_yticklabels(top_20_neg.index, fontsize=10)
ax2.set_xlabel('Correlation with Target', fontsize=12, fontweight='bold')
ax2.set_title('Top 20 Negatively Correlated Features', fontsize=14, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# ✅ Value labels INSIDE bars (left-aligned)
for i, v in enumerate(top_20_neg.values):
    ax2.text(v + 0.002, i, f'{v:.4f}', va='center', ha='left', 
             fontsize=9, color='white', fontweight='bold')

plt.savefig('top_20_correlations_grayscale.png', dpi=300, bbox_inches='tight')
plt.show()

# Feature vs Feature

In [ ]:
# Feature vs Feature Correlation
correlation_matrix = df_eda[feature_cols].corr()
print(correlation_matrix.iloc[:10, :10].round(3))

# Flatten matrix into Series (remove diagonal=1, and duplicates)
corr_flat = correlation_matrix.where(~np.eye(correlation_matrix.shape[0], dtype=bool))
corr_flat = corr_flat.unstack().dropna()

# ABS version
abs_corr_flat = corr_flat.abs()

print("\n📊 Correlation Statistics (Feature–Feature):")
print(f"   Max correlation: {corr_flat.max():.4f}")
print(f"   Min correlation: {corr_flat.min():.4f}")
print(f"   Max |correlation|: {abs_corr_flat.max():.4f}")
print(f"   Mean |correlation|: {abs_corr_flat.mean():.4f}")
print(f"   Median |correlation|: {abs_corr_flat.median():.4f}")

# Categorize by correlation strength
print("\n📊 Number of Feature Pairs by |r| Strength:")
print(f"   |r| ≥ 0.9 (Very Strong): {(abs_corr_flat >= 0.9).sum()}")
print(f"   0.7 ≤ |r| < 0.9 (Strong): {((abs_corr_flat >= 0.7) & (abs_corr_flat < 0.9)).sum()}")
print(f"   0.3 ≤ |r| < 0.7 (Moderate): {((abs_corr_flat >= 0.3) & (abs_corr_flat < 0.7)).sum()}")
print(f"   0 < |r| < 0.3 (Weak): {((abs_corr_flat > 0) & (abs_corr_flat < 0.3)).sum()}")


In [ ]:
# Feature-Feature Correlation Heatmap (Top 50)
top_50_features = pearson_correlations.abs().sort_values(ascending=False).head(50).index.tolist()
corr_subset = correlation_matrix.loc[top_50_features, top_50_features]

import matplotlib.colors as mcolors

# Create custom colormap
colors = ['#2C2C2C', '#6E6E6E', '#B8B8B8', 'white', '#B8B8B8', '#6E6E6E', '#2C2C2C']
n_bins = 100
cmap = mcolors.LinearSegmentedColormap.from_list('custom_gray', colors, N=n_bins)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(corr_subset,
            cmap=cmap,
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1,
            xticklabels=True,
            yticklabels=True,
            ax=ax)

# Add border around the heatmap
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_edgecolor('black')

plt.title('Feature-Feature Correlation Matrix (Top 50 Features)', 
          fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('correlation_heatmap_grayscale.png', dpi=300, bbox_inches='tight')
plt.show()

# DISTRIBUTION

In [ ]:
# ==========================================
# DISTRIBUTION ANALYSIS - ALL FEATURES
# ==========================================
print("\n" + "="*80)
print("DISTRIBUTION ANALYSIS - ALL FEATURES")
print("="*80)

# Get ALL feature columns
feature_cols = [col for col in df_eda.columns if col.startswith('var_')]

print(f"\n📊 Total feature columns analyzed: {len(feature_cols)}")

# Analyze ALL features (not just those with missing values)
distribution_summary_all = []

for col in feature_cols:
    data = df_eda[col].dropna()
    
    if len(data) == 0:
        continue
    
    # Calculate statistics
    skewness = stats.skew(data)
    mean_val = data.mean()
    median_val = data.median()
    mean_median_diff = abs(mean_val - median_val)
    missing_count = df_eda[col].isnull().sum()
    missing_pct = (missing_count / len(df_eda)) * 100
    
    # Determine distribution shape
    if abs(skewness) < 0.5:
        shape = "Symmetric"
        recommendation = "Mean"
    elif abs(skewness) < 1.0:
        shape = "Moderately Skewed"
        recommendation = "Median (safer)"
    else:
        shape = "Highly Skewed"
        recommendation = "Median"
    
    distribution_summary_all.append({
        'Column': col,
        'Missing_Count': missing_count,
        'Missing_%': missing_pct,
        'Mean': mean_val,
        'Median': median_val,
        'Mean-Median_Diff': mean_median_diff,
        'Skewness': skewness,
        'Distribution': shape,
        'Recommendation': recommendation
    })

# Create summary DataFrame
dist_df_all = pd.DataFrame(distribution_summary_all)

# ==========================================
# SUMMARY STATISTICS
# ==========================================
print("\n" + "="*80)
print("SKEWNESS SUMMARY - ALL FEATURES")
print("="*80)

# Count by distribution type
symmetric_count = (dist_df_all['Distribution'] == 'Symmetric').sum()
moderate_skew_count = (dist_df_all['Distribution'] == 'Moderately Skewed').sum()
high_skew_count = (dist_df_all['Distribution'] == 'Highly Skewed').sum()

total_features = len(dist_df_all)

print(f"\n📊 Distribution Breakdown:")
print(f"   Symmetric (|skewness| < 0.5):        {symmetric_count:3d} features ({symmetric_count/total_features*100:5.1f}%)")
print(f"   Moderately Skewed (0.5-1.0):         {moderate_skew_count:3d} features ({moderate_skew_count/total_features*100:5.1f}%)")
print(f"   Highly Skewed (|skewness| > 1.0):    {high_skew_count:3d} features ({high_skew_count/total_features*100:5.1f}%)")
print(f"   {'─'*60}")
print(f"   TOTAL:                                {total_features:3d} features")

print(f"\n📊 Imputation Recommendation (for features with missing values):")
mean_recommendation = (dist_df_all['Recommendation'] == 'Mean').sum()
median_recommendation = (dist_df_all['Recommendation'] != 'Mean').sum()

print(f"   Mean imputation suitable:   {mean_recommendation:3d} features ({mean_recommendation/total_features*100:5.1f}%)")
print(f"   Median imputation safer:    {median_recommendation:3d} features ({median_recommendation/total_features*100:5.1f}%)")

# Skewness statistic

In [ ]:
# ==========================================
# DISTRIBUTION ANALYSIS FOR IMPUTATION
# ==========================================
print("\n" + "-"*80)
print("DISTRIBUTION ANALYSIS FOR IMPUTATION METHOD")
print("-"*80)

# Get columns with missing values
missing_cols = df_eda[feature_cols].columns[df_eda[feature_cols].isnull().any()].tolist()

print(f"\n📊 Feature columns: {len(feature_cols)}")
print(f"📊 Columns with missing values: {len(missing_cols)}")

if len(missing_cols) == 0:
    print("✅ No missing values found in feature columns!")
    print("   Skipping imputation analysis")
else:
    # Create a summary of distributions
    distribution_summary = []
    
    for col in missing_cols:
        data = df_eda[col].dropna()
        
        # Calculate statistics
        skewness = stats.skew(data)
        mean_val = data.mean()
        median_val = data.median()
        
        # Determine if distribution is skewed
        if abs(skewness) < 0.5:
            shape = "Symmetric ✅"
            recommendation = "Mean"
        elif abs(skewness) < 1.0:
            shape = "Moderately Skewed ⚠️"
            recommendation = "Median (safer)"
        else:
            shape = "Highly Skewed ❌"
            recommendation = "Median"
        
        distribution_summary.append({
            'Column': col,
            'Mean': mean_val,
            'Median': median_val,
            'Skewness': skewness,
            'Distribution': shape,
            'Recommendation': recommendation
        })
    
    # Create summary DataFrame
    dist_df = pd.DataFrame(distribution_summary)
    
    print("\n" + "="*80)
    print("DISTRIBUTION SUMMARY")
    print("="*80)
    print(dist_df.to_string(index=False))
    


In [ ]:
# Visualization - ONLY Skewness Distribution
print("\n" + "-"*80)
print("CREATING SKEWNESS DISTRIBUTION VISUALIZATION")
print("-"*80)

# Set global style
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Create single plot for skewness distribution
fig, ax = plt.subplots(figsize=(14, 8))

ax.hist(dist_df['Skewness'], bins=30, color='gray', 
        edgecolor='black', alpha=0.7)
ax.axvline(0, color='black', linestyle='-', linewidth=2, label='Zero (Symmetric)')
ax.axvline(-0.5, color='gray', linestyle='--', linewidth=1.5, label='Moderate threshold')
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5)
ax.set_xlabel('Skewness', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Features', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Skewness Values', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('skewness_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Saved: skewness_distribution.png")

print("\n" + "="*80)
print("✅ EDA COMPLETE!")
print("="*80)

# Data Preparation

In [ ]:
print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
# ==========================================
# LOAD SMALL DATASET FOR MODEL DEVELOPMENT
# ==========================================

print("\n" + "="*80)
print("LOADING SMALL DATASET FOR MODEL DEVELOPMENT")
print("="*80)

# Load small dataset
df = pd.read_csv("data/datafile_small.csv")

print(f"\n📊 Dataset loaded: datafile_small.csv")
print(f"   Purpose: Model experimentation and development")
print(f"   Shape: {df.shape}")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")

print(f"\n✅ Quick check:")
print(f"   Target column exists: {'target' in df.columns}")
print(f"   ID_code exists: {'ID_code' in df.columns}")
print(f"   Feature columns (var_*): {len([col for col in df.columns if col.startswith('var_')])}")

print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"\nClass proportions:")
target_prop = df['target'].value_counts(normalize=True)
print(f"   Class 0: {target_prop[0]*100:.2f}%")
print(f"   Class 1: {target_prop[1]*100:.2f}%")


In [ ]:
# ==========================================
# DATA PREPROCESSING
# ==========================================

print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
# ==========================================
# STEP 1: DROP ID_CODE
# ==========================================

print("\n" + "-"*80)
print("STEP 1: DROP ID_CODE")
print("-"*80)

if 'ID_code' in df.columns:
    df = df.drop(columns=['ID_code'])
    print("✅ Dropped ID_code column")
    print(f"   Reason: Identifier column, not a predictive feature")
else:
    print("⚠️  ID_code column not found (already dropped)")

print(f"\nCurrent shape: {df.shape}")

# ==========================================
# STEP 2: DROP VAR_45 (HIGH MISSING VALUES)
# ==========================================

print("\n" + "-"*80)
print("STEP 2: DROP VAR_45 (HIGH MISSING VALUES)")
print("-"*80)

if 'var_45' in df.columns:
    missing_count = df['var_45'].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    print(f"\nvar_45 statistics:")
    print(f"   Missing values: {missing_count:,}")
    print(f"   Missing percentage: {missing_pct:.2f}%")
    
    df = df.drop(columns=['var_45'])
    print(f"\n✅ Dropped var_45 column")
    print(f"   Reason: {missing_pct:.1f}% missing (>40% threshold)")
else:
    print("⚠️  var_45 column not found (already dropped)")

print(f"\nCurrent shape: {df.shape}")

In [ ]:
# ==========================================
# STEP 3: VERIFY REMAINING FEATURES
# ==========================================

print("\n" + "-"*80)
print("STEP 3: VERIFY REMAINING FEATURES")
print("-"*80)

# Get feature columns (after dropping ID and var_45)
feature_cols = [col for col in df.columns if col.startswith('var_')]

print(f"\nFeature columns remaining: {len(feature_cols)}")
print(f"Target column: {'target' in df.columns}")

# Check remaining missing values
remaining_missing = df[feature_cols].isnull().sum().sum()
print(f"\nRemaining missing values in features: {remaining_missing:,}")

if remaining_missing > 0:
    missing_cols = df[feature_cols].columns[df[feature_cols].isnull().any()].tolist()
    print(f"Columns with missing values: {len(missing_cols)}")
    print(f"\nSummary (top 10):")
    for col in missing_cols[:10]:
        miss_count = df[col].isnull().sum()
        miss_pct = (miss_count / len(df)) * 100
        print(f"   {col}: {miss_count:,} ({miss_pct:.2f}%)")
    if len(missing_cols) > 10:
        print(f"   ... and {len(missing_cols) - 10} more")

In [ ]:
# ==========================================
# STEP 4: PREPARE X AND Y
# ==========================================

print("\n" + "-"*80)
print("STEP 4: PREPARE FEATURES (X) AND TARGET (y)")
print("-"*80)

X = df.drop(columns=["target"])   
y = df["target"]

print(f"\n✅ Features and target separated")
print(f"\nX (Features):")
print(f"   Shape: {X.shape}")
print(f"   Columns: {X.shape[1]}")

print(f"\ny (Target):")
print(f"   Shape: {y.shape}")
print(f"   Unique values: {y.unique()}")

print(f"\nTarget distribution:")
target_counts = y.value_counts()
print(f"   Class 0: {target_counts[0]:,} ({target_counts[0]/len(y)*100:.2f}%)")
print(f"   Class 1: {target_counts[1]:,} ({target_counts[1]/len(y)*100:.2f}%)")
print(f"   Imbalance ratio: {target_counts[0]/target_counts[1]:.2f}:1")

In [ ]:
# ==========================================
# STEP 5: TRAIN-TEST SPLIT
# ==========================================

print("\n" + "-"*80)
print("STEP 5: TRAIN-TEST SPLIT")
print("-"*80)

randomseed = 30

print("\nSplit configuration:")
print(f"   Test size: 20%")
print(f"   Random state: {randomseed}")
print(f"   Stratify: Yes")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=randomseed,
    stratify=y
)

print(f"\n✅ Split complete!")

print(f"\nTraining set:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   Samples: {X_train.shape[0]:,}")

print(f"\nTest set:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   Samples: {X_test.shape[0]:,}")

print(f"\nClass distribution:")
print(f"   Training - Class 0: {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.2f}%)")
print(f"   Training - Class 1: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.2f}%)")
print(f"   Test - Class 0: {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.2f}%)")
print(f"   Test - Class 1: {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.2f}%)")


In [ ]:
# ==========================================
# STEP 6: IMPUTE MISSING VALUES
# ==========================================

print("\n" + "-"*80)
print("STEP 6: IMPUTE MISSING VALUES")
print("-"*80)

print("\nImputation strategy: MEAN")
print("⚠️  Fitted on TRAINING data only!")

# Check missing before
train_missing_before = X_train.isnull().sum().sum()
test_missing_before = X_test.isnull().sum().sum()

print(f"\nBefore imputation:")
print(f"   Training missing: {train_missing_before:,}")
print(f"   Test missing: {test_missing_before:,}")

if train_missing_before > 0 or test_missing_before > 0:
    imputer = SimpleImputer(strategy='mean')
    
    # Fit and transform training data
    X_train = imputer.fit_transform(X_train)
    
    # Transform test data (using training means)
    X_test = imputer.transform(X_test)
    
    # Convert back to DataFrame
    X_train = pd.DataFrame(X_train, columns=df.drop(columns=["target"]).columns)
    X_test = pd.DataFrame(X_test, columns=df.drop(columns=["target"]).columns)
    
    print(f"\n✅ Imputation complete!")
    print(f"\nAfter imputation:")
    print(f"   Training missing: {X_train.isnull().sum().sum()}")
    print(f"   Test missing: {X_test.isnull().sum().sum()}")
else:
    print(f"\n✅ No missing values - skipping imputation")

In [ ]:
# ==========================================
# STEP 7: FINAL VERIFICATION
# ==========================================

print("\n" + "-"*80)
print("STEP 7: FINAL VERIFICATION")
print("-"*80)

print(f"\n✅ PREPROCESSING COMPLETE!")

print(f"\n" + "="*60)
print("FINAL DATASET SUMMARY")
print("="*60)

print(f"\nDataset: datafile_small.csv")

print(f"\nTraining Set:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   Missing values: {X_train.isnull().sum().sum()}")

print(f"\nTest Set:")
print(f"   X_test: {X_test.shape}")
print(f"   y_test: {y_test.shape}")
print(f"   Missing values: {X_test.isnull().sum().sum()}")

print(f"\nClass Distribution:")
print(f"   Train - Class 0: {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.2f}%)")
print(f"   Train - Class 1: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.2f}%)")
print(f"   Test - Class 0: {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.2f}%)")
print(f"   Test - Class 1: {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.2f}%)")

print(f"\nQuality Checks:")
print(f"   ✓ No missing values")
print(f"   ✓ Stratified split maintained")
print(f"   ✓ Imputation on training data only")

print("\n" + "="*80)
print("DATA READY FOR MODELING!")
print("="*80)

# Modelling

# CatBoost builds trees one by one
More iterations = more trees = more learning

Justification:

1. iterations (Number of trees/boosting rounds)

* CatBoost builds trees one by one
* More iterations = more trees = more learning
* Like studying for an exam: more hours = better learning (up to a point!)

Justification:
Too low (<50): Model doesn't learn enough (underfitting)
Sweet spot (100-200): Good balance
Too high (>300): Takes forever, might overfit, diminishing returns

2. learning_rate (How fast the model learns)
* Controls how much each tree contributes
* Small learning rate = slow, careful learning
* Large learning rate = fast but might miss optimal solution

It's like walking to a target:

Big steps (0.2): Fast but might step over the target
Small steps (0.01): Slow but more accurate
Medium steps (0.1): Good balance!

Justification:
Too low (<0.01): Training takes forever
Sweet spot (0.05-0.15): Good balance of speed and accuracy
Too high (>0.3): Model overshoots, unstable learning

3. depth (Tree depth/complexity)

* How many "if-then" questions each tree can ask
* Depth 4 = 4 levels of questions
* Deeper trees = more complex patterns

Justification:
Too shallow (<3): Can't capture complex patterns (underfitting)
Sweet spot (4-8): Good for most problems
Too deep (>10): Overfits, very slow training

4. l2_leaf_reg (L2 regularization)

* Penalty for making predictions too extreme
* Prevents overfitting by keeping predictions moderate
* Higher values = stronger penalty = simpler model

Justification:

Too low (<1): No regularization, overfitting risk
Sweet spot (1-5): Good balance
Too high (>10): Underfitting, model too simple

5. scale_pos_weight (Class imbalance correction)
* Tells model "pay more attention to minority class"
* Calculated as: (# of class 0) / (# of class 1)
* In your case: ~9 (because you have 90% class 0, 10% class 1)

Justification:

This is not a hyperparameter to tune!
It's calculated from your data
Formula: (majority class count) / (minority class count)

In [ ]:
# ==========================================
# MODEL DEVELOPMENT WITH CATBOOST
# ==========================================

print("\n" + "="*80)
print("MODEL DEVELOPMENT - CATBOOST")
print("="*80)

import time
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report, matthews_corrcoef
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import fbeta_score, average_precision_score


In [ ]:
# ==========================================
# BASELINE: NO BALANCING
# ==========================================

print("\n" + "="*80)
print("METHOD 1: BASELINE (NO BALANCING)")
print("="*80)

start_time = time.time()

model_baseline = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=30,
    verbose=False
)

model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_test)
y_proba_baseline = model_baseline.predict_proba(X_test)[:, 1]

results_baseline = {
    'Method': 'Baseline (No Balancing)',
    'Train_Size': len(y_train),
    'Training_Time': time.time() - start_time,
    'Accuracy': accuracy_score(y_test, y_pred_baseline),
    'Precision': precision_score(y_test, y_pred_baseline),
    'Recall': recall_score(y_test, y_pred_baseline),
    'F1_Score': f1_score(y_test, y_pred_baseline),
    'ROC_AUC': roc_auc_score(y_test, y_proba_baseline),
    'MCC': matthews_corrcoef(y_test, y_pred_baseline) 
}

print(f"\n✅ Baseline trained in {results_baseline['Training_Time']:.2f}s")
print(f"\nResults:")
print(f"   Accuracy:  {results_baseline['Accuracy']:.4f}")
print(f"   Precision: {results_baseline['Precision']:.4f}")
print(f"   Recall:    {results_baseline['Recall']:.4f}")
print(f"   F1-Score:  {results_baseline['F1_Score']:.4f}")
print(f"   ROC-AUC:   {results_baseline['ROC_AUC']:.4f}")
print(f"   MCC:       {results_baseline['MCC']:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nclassification_report:")
print(classification_report(y_test, y_pred_baseline))

In [ ]:
# ==========================================
# BASELINE: NO BALANCING
# ==========================================

print("\n" + "="*80)
print("METHOD 1: BASELINE (NO BALANCING)")
print("="*80)

start_time = time.time()

model_baseline = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=30,
    verbose=False
)

model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_test)
y_proba_baseline = model_baseline.predict_proba(X_test)[:, 1]

results_baseline = {
    'Method': 'Baseline (No Balancing)',
    'Train_Size': len(y_train),
    'Training_Time': time.time() - start_time,
    'Accuracy': accuracy_score(y_test, y_pred_baseline),
    'Precision': precision_score(y_test, y_pred_baseline),
    'Recall': recall_score(y_test, y_pred_baseline),
    'F1_Score': f1_score(y_test, y_pred_baseline),
    'F2_Score': fbeta_score(y_test, y_pred_baseline, beta=2),  # ✅ F2-Score
    'ROC_AUC': roc_auc_score(y_test, y_proba_baseline),
    'PR_AUC': average_precision_score(y_test, y_proba_baseline),  # ✅ PR-AUC
    'MCC': matthews_corrcoef(y_test, y_pred_baseline)
}

print(f"\n✅ Baseline trained in {results_baseline['Training_Time']:.2f}s")

print(f"\nResults:")
print(f"   Accuracy:  {results_baseline['Accuracy']:.4f}")
print(f"   Precision: {results_baseline['Precision']:.4f}")
print(f"   Recall:    {results_baseline['Recall']:.4f}")
print(f"   F1-Score:  {results_baseline['F1_Score']:.4f}")
print(f"   F2-Score:  {results_baseline['F2_Score']:.4f}")  # ✅ Added
print(f"   ROC-AUC:   {results_baseline['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_baseline['PR_AUC']:.4f}")  # ✅ Added
print(f"   MCC:       {results_baseline['MCC']:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_baseline))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_baseline))

In [ ]:
# ==========================================
# METHOD 2: CLASS WEIGHTS
# ==========================================

print("\n" + "="*80)
print("METHOD 2: CLASS WEIGHTS")
print("="*80)

start_time = time.time()

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

model_weighted = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=30,
    verbose=False
)

model_weighted.fit(X_train, y_train)

y_pred_weighted = model_weighted.predict(X_test)
y_proba_weighted = model_weighted.predict_proba(X_test)[:, 1]

results_weighted = {
    'Method': 'Class Weights',
    'Train_Size': len(y_train),
    'Training_Time': time.time() - start_time,
    'Accuracy': accuracy_score(y_test, y_pred_weighted),
    'Precision': precision_score(y_test, y_pred_weighted),
    'Recall': recall_score(y_test, y_pred_weighted),
    'F1_Score': f1_score(y_test, y_pred_weighted),
    'F2_Score': fbeta_score(y_test, y_pred_weighted, beta=2),  
    'ROC_AUC': roc_auc_score(y_test, y_proba_weighted),
    'PR_AUC': average_precision_score(y_test, y_proba_weighted),  
    'MCC': matthews_corrcoef(y_test, y_pred_weighted)
}

print(f"\n✅ Class Weights trained in {results_weighted['Training_Time']:.2f}s")

print(f"\nResults:")
print(f"   Accuracy:  {results_weighted['Accuracy']:.4f}")
print(f"   Precision: {results_weighted['Precision']:.4f}")
print(f"   Recall:    {results_weighted['Recall']:.4f}")
print(f"   F1-Score:  {results_weighted['F1_Score']:.4f}")
print(f"   F2-Score:  {results_weighted['F2_Score']:.4f}") 
print(f"   ROC-AUC:   {results_weighted['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_weighted['PR_AUC']:.4f}")  
print(f"   MCC:       {results_weighted['MCC']:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_weighted))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_weighted))

In [ ]:
# ==========================================
# METHOD 3: SMOTE
# ==========================================

print("\n" + "="*80)
print("METHOD 3: SMOTE (OVERSAMPLING)")
print("="*80)

start_time = time.time()

smote = SMOTE(random_state=30)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nBefore SMOTE: {len(y_train):,}")
print(f"After SMOTE:  {len(y_train_smote):,}")
print(f"Class balance: {(y_train_smote==0).sum():,} vs {(y_train_smote==1).sum():,}")

model_smote = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=30,
    verbose=False
)

model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)
y_proba_smote = model_smote.predict_proba(X_test)[:, 1]

results_smote = {
    'Method': 'SMOTE',
    'Train_Size': len(y_train_smote),
    'Training_Time': time.time() - start_time,
    'Accuracy': accuracy_score(y_test, y_pred_smote),
    'Precision': precision_score(y_test, y_pred_smote),
    'Recall': recall_score(y_test, y_pred_smote),
    'F1_Score': f1_score(y_test, y_pred_smote),
    'F2_Score': fbeta_score(y_test, y_pred_smote, beta=2),  
    'ROC_AUC': roc_auc_score(y_test, y_proba_smote),
    'PR_AUC': average_precision_score(y_test, y_proba_smote),  
    'MCC': matthews_corrcoef(y_test, y_pred_smote)
}

print(f"\n✅ SMOTE trained in {results_smote['Training_Time']:.2f}s")

print(f"\nResults:")
print(f"   Accuracy:  {results_smote['Accuracy']:.4f}")
print(f"   Precision: {results_smote['Precision']:.4f}")
print(f"   Recall:    {results_smote['Recall']:.4f}")
print(f"   F1-Score:  {results_smote['F1_Score']:.4f}")
print(f"   F2-Score:  {results_smote['F2_Score']:.4f}")  
print(f"   ROC-AUC:   {results_smote['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_smote['PR_AUC']:.4f}")  
print(f"   MCC:       {results_smote['MCC']:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_smote))

In [ ]:
# ==========================================
# METHOD 4: EASY ENSEMBLE + CATBOOST (MULTIPLE VERSIONS)
# ==========================================
print("\n" + "="*80)
print("METHOD 4: EASY ENSEMBLE + CATBOOST - TESTING DIFFERENT N_ESTIMATORS")
print("="*80)

from imblearn.ensemble import EasyEnsembleClassifier
from catboost import CatBoostClassifier

print("\nEasy Ensemble + CatBoost strategy:")
print("   • Creates multiple balanced subsets via undersampling")
print("   • Trains CatBoost models on each subset")
print("   • Combines predictions via voting")
print("   • Uses more data than single undersampling")

# Test different n_estimators
n_estimators_list = [10, 25, 100, 500, 1000]

# Store results separately
results_eec_10 = None
results_eec_25 = None
results_eec_100 = None
results_eec_500 = None
results_eec_1000 = None

for n_est in n_estimators_list:
    print("\n" + "-"*80)
    print(f"TESTING: n_estimators = {n_est}")
    print("-"*80)
    
    start_time = time.time()
    
    eec = EasyEnsembleClassifier(
        estimator=CatBoostClassifier(
            iterations=100,
            depth=6,
            learning_rate=0.1,
            random_state=30,
            verbose=False
        ),
        n_estimators=n_est,
        random_state=30,
        n_jobs=-1
    )
    
    print(f"\nConfiguration:")
    print(f"   n_estimators: {n_est} (creates {n_est} balanced subsets)")
    print(f"   base_estimator: CatBoostClassifier")
    print(f"   sampling_strategy: auto (balance to 50/50)")
    
    eec.fit(X_train, y_train)
    
    y_pred_eec = eec.predict(X_test)
    y_proba_eec = eec.predict_proba(X_test)[:, 1]
    
    results_eec = {
        'Method': f'Easy Ensemble + CatBoost (n={n_est})',
        'N_Estimators': n_est,
        'Train_Size': len(y_train),
        'Training_Time': time.time() - start_time,
        'Accuracy': accuracy_score(y_test, y_pred_eec),
        'Precision': precision_score(y_test, y_pred_eec),
        'Recall': recall_score(y_test, y_pred_eec),
        'F1_Score': f1_score(y_test, y_pred_eec),
        'F2_Score': fbeta_score(y_test, y_pred_eec, beta=2),
        'ROC_AUC': roc_auc_score(y_test, y_proba_eec),
        'PR_AUC': average_precision_score(y_test, y_proba_eec),
        'MCC': matthews_corrcoef(y_test, y_pred_eec)
    }
    
    # Store in separate variables
    if n_est == 10:
        results_eec_10 = results_eec
    elif n_est == 25:
        results_eec_25 = results_eec
    elif n_est == 100:
        results_eec_100 = results_eec
    elif n_est == 500:
        results_eec_500 = results_eec
    elif n_est == 1000:
        results_eec_1000 = results_eec
    
    print(f"\n✅ Easy Ensemble + CatBoost (n={n_est}) trained in {results_eec['Training_Time']:.2f}s")
    
    print(f"\nResults:")
    print(f"   Accuracy:  {results_eec['Accuracy']:.4f}")
    print(f"   Precision: {results_eec['Precision']:.4f}")
    print(f"   Recall:    {results_eec['Recall']:.4f}")
    print(f"   F1-Score:  {results_eec['F1_Score']:.4f}")
    print(f"   F2-Score:  {results_eec['F2_Score']:.4f}")
    print(f"   ROC-AUC:   {results_eec['ROC_AUC']:.4f}")
    print(f"   PR-AUC:    {results_eec['PR_AUC']:.4f}")
    print(f"   MCC:       {results_eec['MCC']:.4f}")
    
    print(f"\nConfusion Matrix:")
    cm_eec = confusion_matrix(y_test, y_pred_eec)
    print(cm_eec)
    print(f"\n   True Negatives:  {cm_eec[0, 0]:,}")
    print(f"   False Positives: {cm_eec[0, 1]:,}")
    print(f"   False Negatives: {cm_eec[1, 0]:,}")
    print(f"   True Positives:  {cm_eec[1, 1]:,}")

In [ ]:
# ==========================================
# EASY ENSEMBLE n_estimators SENSITIVITY ANALYSIS
# ==========================================
print("\n" + "="*80)
print("APPENDIX: Easy Ensemble n_estimators Selection Analysis")
print("="*80)

# n값들 비교
n_comparison_results = [
    results_eec_10,
    results_eec_25,
    results_eec_100,
    results_eec_500,
    results_eec_1000
]

n_comparison_df = pd.DataFrame(n_comparison_results)

# Extract n_estimators from method name
n_comparison_df['n_estimators'] = [10, 25, 100, 500, 1000]

# Calculate efficiency metrics
n_comparison_df['Performance_vs_Best'] = (n_comparison_df['PR_AUC'] / 
                                           n_comparison_df['PR_AUC'].max() * 100)
n_comparison_df['Time_vs_Worst'] = (n_comparison_df['Training_Time'] / 
                                     n_comparison_df['Training_Time'].max() * 100)
n_comparison_df['Efficiency_Score'] = (n_comparison_df['Performance_vs_Best'] / 
                                        n_comparison_df['Time_vs_Worst'])

print("\n📈 n_estimators Comparison:")
display_n = n_comparison_df[['n_estimators', 'Training_Time', 'PR_AUC', 
                              'F2_Score', 'MCC', 'Performance_vs_Best', 
                              'Time_vs_Worst', 'Efficiency_Score']].copy()

for col in ['PR_AUC', 'F2_Score', 'MCC']:
    display_n[col] = display_n[col].apply(lambda x: f"{x:.4f}")
for col in ['Training_Time']:
    display_n[col] = display_n[col].apply(lambda x: f"{x:.2f}s")
for col in ['Performance_vs_Best', 'Time_vs_Worst', 'Efficiency_Score']:
    display_n[col] = display_n[col].apply(lambda x: f"{x:.2f}")

print(display_n.to_string(index=False))

print("\n🎯 Selection Rationale:")
print(f"   • n=10 achieves {n_comparison_df.loc[0, 'Performance_vs_Best']:.1f}% of best performance")
print(f"   • n=10 requires only {n_comparison_df.loc[0, 'Time_vs_Worst']:.1f}% of max training time")
print(f"   • n=10 has highest efficiency score: {n_comparison_df.loc[0, 'Efficiency_Score']:.2f}")
print(f"   • Aligned with imbalance ratio (~9:1) as per Liu et al. (2008)")

print("\n✅ Conclusion: n=10 selected for main comparison")

In [ ]:
# ==========================================
# SUMMARY TABLE
# ==========================================

print("\n" + "="*80)
print("EASY ENSEMBLE COMPARISON - ALL N_ESTIMATORS")
print("="*80)

eec_comparison_df = pd.DataFrame([
    results_eec_10,
    results_eec_25,
    results_eec_100,
    results_eec_500,
    results_eec_1000
])

# Format the DataFrame with 4 decimal places
print("\n📊 All Easy Ensemble Results:")
display_df = eec_comparison_df[['Method', 'N_Estimators', 'Training_Time', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']].copy()

# Format numeric columns to 4 decimal places
for col in ['Training_Time', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

print(display_df.to_string(index=False))

print("\n✅ Results stored in separate variables:")
print("   • results_eec_10")
print("   • results_eec_25")
print("   • results_eec_100")
print("   • results_eec_500")
print("   • results_eec_1000")

print("\n" + "="*80)
print("EASY ENSEMBLE TESTING COMPLETE!")
print("="*80)

Justification
Performance: Only 0.55% worse than n=1000 (negligible)
Efficiency: 10× faster training
Theory: 100 estimators sufficient for stability (Breiman, 1996)
Hyperparameter tuning: Makes Grid Search/Optuna feasible
Deployment: Smaller, faster, easier to maintain

In [ ]:
# ==========================================
# COMPARE ALL METHODS
# ==========================================
print("\n" + "="*80)
print("RESULTS COMPARISON - ALL METHODS")
print("="*80)

# Include only core methods for comparison
all_results = [
    results_baseline,
    results_weighted,
    results_smote,
    results_eec_10      # ✅ Only Easy Ensemble n=10
]

results_df = pd.DataFrame(all_results)

# Sort by PR-AUC (primary), then F2-Score, then MCC
results_df_sorted = results_df.sort_values(
    by=['PR_AUC', 'F2_Score', 'MCC'],
    ascending=[False, False, False]
).reset_index(drop=True)

print("\n📊 All Models Ranked by PR-AUC → F2-Score → MCC:")
display_cols = ['Method', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']

# Format for display
display_df = results_df_sorted[display_cols].copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

print(display_df.to_string(index=False))

# Alternative views
print("\n" + "="*80)
print("📊 RANKED BY DIFFERENT METRICS")
print("="*80)

# By PR-AUC
print("\n🔹 Ranked by PR-AUC (PRIMARY - imbalanced data focus):")
results_by_prauc = results_df.sort_values('PR_AUC', ascending=False)
display_prauc = results_by_prauc[['Method', 'PR_AUC', 'F2_Score', 'MCC', 'F1_Score', 'ROC_AUC']].copy()
for col in ['PR_AUC', 'F2_Score', 'MCC', 'F1_Score', 'ROC_AUC']:
    display_prauc[col] = display_prauc[col].apply(lambda x: f"{x:.4f}")
print(display_prauc.to_string(index=False))

# By F2-Score
print("\n🔹 Ranked by F2-Score (SECONDARY - favors Recall):")
results_by_f2 = results_df.sort_values('F2_Score', ascending=False)
display_f2 = results_by_f2[['Method', 'F2_Score', 'PR_AUC', 'Recall', 'F1_Score', 'MCC']].copy()
for col in ['F2_Score', 'PR_AUC', 'Recall', 'F1_Score', 'MCC']:
    display_f2[col] = display_f2[col].apply(lambda x: f"{x:.4f}")
print(display_f2.to_string(index=False))

# By MCC
print("\n🔹 Ranked by MCC (TERTIARY - balanced metric):")
results_by_mcc = results_df.sort_values('MCC', ascending=False)
display_mcc = results_by_mcc[['Method', 'MCC', 'PR_AUC', 'F2_Score', 'F1_Score', 'Accuracy']].copy()
for col in ['MCC', 'PR_AUC', 'F2_Score', 'F1_Score', 'Accuracy']:
    display_mcc[col] = display_mcc[col].apply(lambda x: f"{x:.4f}")
print(display_mcc.to_string(index=False))

print("\n" + "="*80)
print("✅ COMPARISON COMPLETE!")
print("="*80)

print("\n🎯 Model Selection Priority:")
print("   1️⃣ PR-AUC (Primary) - Best for imbalanced data")
print("   2️⃣ F2-Score (Secondary) - Favors Recall over Precision")
print("   3️⃣ MCC (Tertiary) - Balanced correlation metric")

print("\n📋 Methods Compared:")
print("   1. Baseline (No Balancing)")
print("   2. Class Weights")
print("   3. SMOTE (Oversampling)")
print("   4. Easy Ensemble + CatBoost (n=10, Undersampling)")

print("\n💡 Easy Ensemble Configuration:")
print("   • n_estimators=10 (optimal efficiency)")
print("   • 97.7% of n=1000 performance")
print("   • 48× faster training time")
print("   • Theoretically justified (imbalance ratio ~9:1)")

### 4.3 Model Comparison Framework

We compared four distinct approaches to handling class imbalance 
in credit application prediction:

1. **Baseline (No Balancing)**: Standard CatBoost classifier without 
   any balancing technique, serving as the performance floor.

2. **Class Weights**: Cost-sensitive learning approach where 
   misclassification costs are weighted inversely proportional to 
   class frequencies.

3. **SMOTE (Synthetic Minority Over-sampling Technique)**: 
   Oversampling method that generates synthetic minority class 
   samples using k-nearest neighbors interpolation.

4. **Easy Ensemble + CatBoost**: Undersampling ensemble method that 
   creates multiple balanced subsets and aggregates predictions 
   from independent CatBoost models.

For Easy Ensemble, we selected n_estimators=10 based on empirical 
analysis showing it achieves 97.7% of maximum performance while 
requiring only 2.1% of computational resources compared to n=1000 
(detailed sensitivity analysis in Appendix A).

Models were evaluated using three complementary metrics prioritized 
for imbalanced classification tasks:

- **Primary**: PR-AUC (Precision-Recall Area Under Curve) - most 
  appropriate for imbalanced datasets as it focuses on positive 
  class performance


# Feature Selection

In [ ]:
# ==========================================
# FEATURE SELECTION - CATBOOST BUILT-IN
# ==========================================
print("\n" + "="*80)
print("FEATURE SELECTION - CATBOOST BUILT-IN METHOD")
print("="*80)

print("\n🎯 Using CatBoost's select_features() method")
print("   • Evaluates feature importance during training")
print("   • Uses loss function optimization")
print("   • More sophisticated than simple importance filtering")

# ==========================================
# Step 0: Determine best baseline model for feature selection
# ==========================================
print("\n📊 Step 0: Selecting best balancing method as baseline...")

# Find best model by PR-AUC
best_model_name = results_df.loc[results_df['PR_AUC'].idxmax(), 'Method']
best_model_prauc = results_df['PR_AUC'].max()

print(f"\n🏆 Best balancing method: {best_model_name}")
print(f"   PR-AUC: {best_model_prauc:.4f}")

print(f"\n📊 Using Easy Ensemble + CatBoost (n=10) configuration as baseline")
print(f"   Base model: CatBoost with iterations=100, depth=6, learning_rate=0.1")
print(f"   n_estimators=10 (optimal efficiency)")
print(f"   No scale_pos_weight (Easy Ensemble already balances data)")

In [ ]:
# ==========================================
# Step 1: Run feature selection
# ==========================================
print("\n📊 Step 1: Running feature selection...")
start_time = time.time()

# Create a model for feature selection
# Note: We don't use scale_pos_weight because Easy Ensemble already balances data
model_for_selection = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_state=30,
    verbose=False
)

# Run feature selection
from catboost import Pool
train_pool = Pool(X_train, y_train)

# Select features using CatBoost's built-in method
summary = model_for_selection.select_features(
    train_pool,
    features_for_select=list(range(X_train.shape[1])),
    num_features_to_select=X_train.shape[1] // 2,  # Select top 50% features
    steps=3,
    algorithm='RecursiveByShapValues',
    shap_calc_type='Regular',
    train_final_model=False,
    verbose=False,
    plot=False
)

selection_time = time.time() - start_time
print(f"\n✅ Feature selection completed in {selection_time:.2f}s")

# Get selected features
selected_features_names = summary['selected_features_names']
n_features_selected = len(selected_features_names)

print(f"\n📊 Feature Selection Results:")
print(f"   Original features: {X_train.shape[1]}")
print(f"   Selected features: {n_features_selected}")
print(f"   Features removed: {X_train.shape[1] - n_features_selected}")
print(f"   Reduction: {(1 - n_features_selected/X_train.shape[1])*100:.1f}%")
print(f"   Algorithm: RecursiveByShapValues, 3 steps, Regular SHAP calculation")

print(f"\nTop 20 selected features:")
for i, feat in enumerate(selected_features_names[:20], 1):
    print(f"   {i}. {feat}")
if len(selected_features_names) > 20:
    print(f"   ... and {len(selected_features_names) - 20} more")

In [ ]:
# ==========================================
# Step 2: Transform datasets
# ==========================================
print("\n📊 Step 2: Transforming datasets with selected features...")

# Use feature names directly to select columns
X_train_selected = X_train[selected_features_names]
X_test_selected = X_test[selected_features_names]

print(f"   X_train_selected shape: {X_train_selected.shape}")
print(f"   X_test_selected shape: {X_test_selected.shape}")

# ==========================================
# Step 3: Train Easy Ensemble + CatBoost (n=10) WITH selected features
# ==========================================
print("\n📊 Step 3: Training Easy Ensemble + CatBoost (n=10) WITH selected features...")

start_time = time.time()

eec_10_fs = EasyEnsembleClassifier(
    estimator=CatBoostClassifier(
        iterations=100,
        depth=6,
        learning_rate=0.1,
        random_state=30,
        verbose=False
    ),
    n_estimators=10,  # ✅ Using n=10 for efficiency
    random_state=30,
    n_jobs=-1
)

eec_10_fs.fit(X_train_selected, y_train)

y_pred_eec_10_fs = eec_10_fs.predict(X_test_selected)
y_proba_eec_10_fs = eec_10_fs.predict_proba(X_test_selected)[:, 1]

training_time_10_fs = time.time() - start_time

results_eec_10_fs = {
    'Method': 'Easy Ensemble + CatBoost (n=10) + Feature Selection',
    'N_Estimators': 10,
    'N_Features': n_features_selected,
    'Train_Size': len(y_train),
    'Training_Time': training_time_10_fs,
    'Accuracy': accuracy_score(y_test, y_pred_eec_10_fs),
    'Precision': precision_score(y_test, y_pred_eec_10_fs),
    'Recall': recall_score(y_test, y_pred_eec_10_fs),
    'F1_Score': f1_score(y_test, y_pred_eec_10_fs),
    'F2_Score': fbeta_score(y_test, y_pred_eec_10_fs, beta=2),
    'ROC_AUC': roc_auc_score(y_test, y_proba_eec_10_fs),
    'PR_AUC': average_precision_score(y_test, y_proba_eec_10_fs),
    'MCC': matthews_corrcoef(y_test, y_pred_eec_10_fs)
}

print(f"\n✅ Model trained in {training_time_10_fs:.2f}s")

print(f"\nResults:")
print(f"   Accuracy:  {results_eec_10_fs['Accuracy']:.4f}")
print(f"   Precision: {results_eec_10_fs['Precision']:.4f}")
print(f"   Recall:    {results_eec_10_fs['Recall']:.4f}")
print(f"   F1-Score:  {results_eec_10_fs['F1_Score']:.4f}")
print(f"   F2-Score:  {results_eec_10_fs['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_eec_10_fs['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_eec_10_fs['PR_AUC']:.4f}")
print(f"   MCC:       {results_eec_10_fs['MCC']:.4f}")

print(f"\nConfusion Matrix:")
cm_eec_10_fs = confusion_matrix(y_test, y_pred_eec_10_fs)
print(cm_eec_10_fs)
print(f"\n   True Negatives:  {cm_eec_10_fs[0, 0]:,}")
print(f"   False Positives: {cm_eec_10_fs[0, 1]:,}")
print(f"   False Negatives: {cm_eec_10_fs[1, 0]:,}")
print(f"   True Positives:  {cm_eec_10_fs[1, 1]:,}")

In [ ]:
# ==========================================
# COMPARISON: WITH vs WITHOUT FEATURE SELECTION (n=10)
# ==========================================
print("\n" + "="*80)
print("COMPARISON: EASY ENSEMBLE + CATBOOST (n=10) WITH vs WITHOUT FEATURE SELECTION")
print("="*80)

comparison_eec10_df = pd.DataFrame([
    {**results_eec_10, 'N_Features': X_train.shape[1]},  # WITHOUT Feature Selection
    results_eec_10_fs  # WITH Feature Selection
])

print("\n📊 Results Comparison:")

# Format for display
display_eec10 = comparison_eec10_df.copy()
for col in ['Training_Time', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']:
    if col in display_eec10.columns:
        display_eec10[col] = display_eec10[col].apply(lambda x: f"{x:.4f}")

print(display_eec10[['Method', 'N_Features', 'Training_Time', 'Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']].to_string(index=False))

# Calculate improvements/changes
print("\n📊 Changes with Feature Selection:")
print(f"   Features: {X_train.shape[1]} → {results_eec_10_fs['N_Features']} ({(1-results_eec_10_fs['N_Features']/X_train.shape[1])*100:.1f}% reduction)")
print(f"   Training Time: {results_eec_10['Training_Time']:.2f}s → {training_time_10_fs:.2f}s ({((training_time_10_fs/results_eec_10['Training_Time'])-1)*100:+.1f}%)")

print(f"\n📊 Metric Changes:")
for metric in ['Accuracy', 'Precision', 'Recall', 'F1_Score', 'F2_Score', 'ROC_AUC', 'PR_AUC', 'MCC']:
    change = results_eec_10_fs[metric] - results_eec_10[metric]
    change_pct = (change / results_eec_10[metric]) * 100
    print(f"   {metric:12s}: {results_eec_10[metric]:.4f} → {results_eec_10_fs[metric]:.4f} ({change:+.4f}, {change_pct:+.2f}%)")

# Determine if feature selection helped
print("\n" + "="*80)
print("VERDICT")
print("="*80)

if results_eec_10_fs['PR_AUC'] > results_eec_10['PR_AUC']:
    improvement = results_eec_10_fs['PR_AUC'] - results_eec_10['PR_AUC']
    improvement_pct = (improvement / results_eec_10['PR_AUC']) * 100
    print(f"\n✅ Feature selection IMPROVED performance!")
    print(f"   PR-AUC: {results_eec_10['PR_AUC']:.4f} → {results_eec_10_fs['PR_AUC']:.4f}")
    print(f"   Improvement: +{improvement:.4f} (+{improvement_pct:.2f}%)")
elif results_eec_10_fs['PR_AUC'] < results_eec_10['PR_AUC']:
    decrease = results_eec_10['PR_AUC'] - results_eec_10_fs['PR_AUC']
    decrease_pct = (decrease / results_eec_10['PR_AUC']) * 100
    print(f"\n⚠️  Feature selection DECREASED performance")
    print(f"   PR-AUC: {results_eec_10['PR_AUC']:.4f} → {results_eec_10_fs['PR_AUC']:.4f}")
    print(f"   Decrease: -{decrease:.4f} (-{decrease_pct:.2f}%)")
else:
    print(f"\n➡️  Feature selection had MINIMAL impact on performance")
    print(f"   PR-AUC: {results_eec_10['PR_AUC']:.4f} (unchanged)")

# Additional insights
print(f"\n📊 Additional Insights:")

# Training time efficiency
time_ratio = training_time_10_fs / results_eec_10['Training_Time']
print(f"   Training time ratio: {time_ratio:.2f}x")
if time_ratio < 1:
    print(f"   ✅ {(1-time_ratio)*100:.1f}% faster with feature selection")
elif time_ratio > 1:
    print(f"   ⚠️  {(time_ratio-1)*100:.1f}% slower with feature selection")
else:
    print(f"   ➡️  Similar training time")

# Feature reduction benefit
feature_reduction = (1 - results_eec_10_fs['N_Features'] / X_train.shape[1]) * 100
print(f"   Feature reduction: {feature_reduction:.1f}%")
print(f"   ✅ Simpler model (reduced overfitting risk)")

# Performance per feature
perf_per_feature_without = results_eec_10['PR_AUC'] / X_train.shape[1]
perf_per_feature_with = results_eec_10_fs['PR_AUC'] / results_eec_10_fs['N_Features']
print(f"   PR-AUC per feature (without FS): {perf_per_feature_without:.6f}")
print(f"   PR-AUC per feature (with FS): {perf_per_feature_with:.6f}")
if perf_per_feature_with > perf_per_feature_without:
    improvement_efficiency = ((perf_per_feature_with / perf_per_feature_without) - 1) * 100
    print(f"   ✅ {improvement_efficiency:.1f}% more efficient feature usage with FS")


# No Feature Selection

**Justification:**

We selected the model WITHOUT feature selection based on PR-AUC 
as our primary evaluation metric. While feature selection reduced 
the feature space by 50.3% and improved F2-Score (+0.77%) and 
MCC (+0.84%), it resulted in a -0.56% decrease in PR-AUC 
(0.4851 → 0.4824).

Given that PR-AUC is the most appropriate metric for imbalanced 
classification tasks (He & Garcia, 2009), we prioritized maximizing 
this metric over model simplicity. The performance difference, 
though small, represents a trade-off we were unwilling to accept 
for the sake of feature reduction.

**Final Configuration:**
- Method: Easy Ensemble + CatBoost (n=10)
- Features: 199 (all features)
- PR-AUC: 0.4851
- F2-Score: 0.5457
- Training Time: 8.02s

# with feature selection

**Justification:**

We selected the model WITH feature selection despite a marginal 
-0.56% decrease in PR-AUC (0.4851 → 0.4824) for the following reasons:

1. **Improved Recall**: +1.38% increase (0.7379 → 0.7481), meaning 
   the model correctly identifies more distressed firms—critical for 
   risk management where false negatives are costly.

2. **Better F2-Score**: +0.77% improvement, which emphasizes Recall 
   over Precision, aligning with our business objective of minimizing 
   missed bankruptcy predictions.

3. **Statistical Significance**: The PR-AUC decrease (-0.0027) falls 
   within typical model variance and is not statistically significant 
   given our dataset size.

4. **Practical Benefits**: 
   - 50.3% feature reduction improves interpretability
   - 22.4% faster training enables more frequent model updates
   - Reduced overfitting risk with simpler model
   - Lower production deployment costs

5. **Model Generalization**: Simpler models with fewer features 
   typically generalize better to unseen data, a critical consideration 
   for production deployment.

**Final Configuration:**
- Method: Easy Ensemble + CatBoost (n=10) + Feature Selection
- Features: 99 (top 50%)
- PR-AUC: 0.4824
- F2-Score: 0.5499
- Recall: 0.7481
- Training Time: 6.22s

### 4.5 Feature Selection Decision

We employed CatBoost's built-in feature selection method 
(RecursiveByShapValues) which reduced the feature space from 
199 to 99 features (50.3% reduction).

#### Performance Trade-off Analysis

**Metric Changes:**
- PR-AUC: 0.4851 → 0.4824 (-0.56%)
- F2-Score: 0.5457 → 0.5499 (+0.77%)
- Recall: 0.7379 → 0.7481 (+1.38%)
- MCC: 0.3465 → 0.3494 (+0.84%)
- Training Time: 8.02s → 6.22s (-22.4%)

#### Decision Rationale

**We selected the model WITH feature selection** based on the 
following considerations:

1. **Recall Prioritization**: The +1.38% improvement in Recall 
   (0.7379 → 0.7481) translates to correctly identifying 4 
   additional distressed firms in our test set. In credit risk 
   assessment, minimizing false negatives (Type II errors) is 
   critical, as the cost of missing a bankruptcy prediction far 
   exceeds the cost of a false alarm.

2. **F2-Score Improvement**: The +0.77% increase in F2-Score 
   confirms that the model better balances Precision and Recall 
   with emphasis on Recall, aligning with our business objective.

3. **Practical Significance vs Statistical Significance**: 
   The -0.56% decrease in PR-AUC (-0.0027 in absolute terms) is 
   within typical model variance and unlikely to be statistically 
   significant. This marginal decrease is offset by substantial 
   gains in other metrics and practical benefits.

4. **Model Simplicity**: Reducing features by 50.3% (199 → 99) 
   provides several advantages:
   - Improved interpretability for stakeholders
   - Reduced overfitting risk (Hastie et al., 2009)
   - Enhanced generalization to unseen data
   - Lower computational requirements for deployment

5. **Computational Efficiency**: 22.4% reduction in training time 
   (8.02s → 6.22s) enables more frequent model retraining, 
   crucial for adapting to evolving credit market conditions.

6. **Multi-Metric Perspective**: While PR-AUC decreased marginally, 
   two other key metrics (F2-Score and MCC) improved, suggesting 
   the model achieves better overall classification performance.

#### Theoretical Support

This decision aligns with the principle of parsimony in machine 
learning (Occam's Razor): when two models achieve comparable 
performance, the simpler model is preferred (Domingos, 1999). 
Feature selection not only reduces complexity but also mitigates 
the curse of dimensionality, potentially improving model 
robustness in production environments (Guyon & Elisseeff, 2003).

#### Conclusion

The feature-selected model (99 features) provides superior 
Recall performance, enhanced model interpretability, and 
improved computational efficiency, while maintaining competitive 
PR-AUC performance. For a production credit risk system where 
identifying potential defaults is paramount, this configuration 
represents the optimal balance between predictive power and 
practical utility.

# Hyper Parameter Tuning for feature selection in Easy Ensemble

In [ ]:
# ==========================================
# HYPERPARAMETER TUNING - EASY ENSEMBLE + CATBOOST (NO FEATURE SELECTION)
# ==========================================
print("\n" + "="*80)
print("HYPERPARAMETER TUNING - EASY ENSEMBLE + CATBOOST (NO FEATURE SELECTION)")
print("="*80)

print(f"\n🏆 Selected Model: Easy Ensemble + CatBoost (n=10) WITHOUT Feature Selection")
print(f"   Baseline PR-AUC: {results_eec_10['PR_AUC']:.4f}")
print(f"   Baseline F2-Score: {results_eec_10['F2_Score']:.4f}")
print(f"   Baseline Recall: {results_eec_10['Recall']:.4f}")
print(f"   Baseline MCC: {results_eec_10['MCC']:.4f}")
print(f"   Now tuning CatBoost base estimator hyperparameters to improve further...")

# ✅ USE ALL FEATURES (NO FEATURE SELECTION)
X_train_best = X_train  # ✅ ALL 199 features
X_test_best = X_test    # ✅ ALL 199 features
y_train_best = y_train
y_test_best = y_test

print(f"\n   Training data shape: {X_train_best.shape}")
print(f"   Testing data shape: {X_test_best.shape}")
print(f"   Features used: {X_train_best.shape[1]} (all features - no selection)")
print(f"   Note: Easy Ensemble creates balanced subsets, so no scale_pos_weight needed")


In [ ]:
# ==========================================
# OPTION 1: GRID SEARCH
# ==========================================
print("\n" + "="*80)
print("OPTION 1: GRID SEARCH - Easy Ensemble + CatBoost (NO FEATURE SELECTION)")
print("="*80)

from sklearn.model_selection import GridSearchCV

# Define parameter grid for CatBoost base estimator
param_grid = {
    'estimator__iterations': [100, 150, 200],
    'estimator__learning_rate': [0.05, 0.1, 0.15],
    'estimator__depth': [4, 6, 8],
    'estimator__l2_leaf_reg': [1, 3, 5]
}

print("\n📋 Parameter grid for CatBoost base estimator:")
for param, values in param_grid.items():
    print(f"   {param}: {values}")

# Calculate total combinations
total_combinations = 3 * 3 * 3 * 3
print(f"\n📊 Search space:")
print(f"   Total combinations: {total_combinations}")
print(f"   n_estimators: 10 (fixed for efficiency)")
print(f"   Total models to train: {total_combinations} × 3 folds × 10 estimators = {total_combinations * 3 * 10:,}")
print(f"   Estimated time: ~30-45 minutes")

# Create GridSearchCV
grid_search_eec = GridSearchCV(
    EasyEnsembleClassifier(
        estimator=CatBoostClassifier(random_state=30, verbose=False),
        n_estimators=10,  # ✅ Using n=10 for efficiency
        random_state=30,
        n_jobs=-1
    ),
    param_grid,
    cv=3,
    scoring='average_precision',  # PR-AUC
    n_jobs=1,  # Set to 1 because EasyEnsemble already uses parallel processing
    verbose=2,
    return_train_score=False
)

print(f"\n🚀 Starting Grid Search...")
print(f"   Optimization metric: PR-AUC (average_precision)")
print(f"   Cross-validation: 3-fold")
print(f"   This will take approximately 30-45 minutes...")

start_time = time.time()
grid_search_eec.fit(X_train_best, y_train_best)
grid_time_eec = time.time() - start_time

print(f"\n✅ Grid Search completed in {grid_time_eec:.2f}s ({grid_time_eec/60:.1f} minutes)")


In [ ]:
# ==========================================
# GRID SEARCH RESULTS
# ==========================================
print(f"\n" + "="*80)
print("GRID SEARCH - BEST PARAMETERS")
print("="*80)

best_params_grid = grid_search_eec.best_params_
print("\n🏆 Best parameters found:")
for param, value in best_params_grid.items():
    print(f"   {param}: {value}")

print(f"\n📊 Cross-validation results:")
print(f"   Best CV PR-AUC: {grid_search_eec.best_score_:.4f}")
print(f"   Baseline PR-AUC: {results_eec_10['PR_AUC']:.4f}")
print(f"   Improvement: {grid_search_eec.best_score_ - results_eec_10['PR_AUC']:.4f} ({((grid_search_eec.best_score_ / results_eec_10['PR_AUC']) - 1) * 100:+.2f}%)")

# Evaluate on test set
y_pred_grid_eec = grid_search_eec.predict(X_test_best)
y_proba_grid_eec = grid_search_eec.predict_proba(X_test_best)[:, 1]

print(f"\n" + "="*80)
print("GRID SEARCH - TEST SET RESULTS")
print("="*80)

results_grid_eec = {
    'Method': 'Easy Ensemble (n=10) + Grid Search (NO FS)',
    'N_Estimators': 10,
    'N_Features': X_train_best.shape[1],
    'Training_Time': grid_time_eec,
    'Best_Params': best_params_grid,
    'CV_PR_AUC': grid_search_eec.best_score_,
    'Accuracy': accuracy_score(y_test_best, y_pred_grid_eec),
    'Precision': precision_score(y_test_best, y_pred_grid_eec),
    'Recall': recall_score(y_test_best, y_pred_grid_eec),
    'F1_Score': f1_score(y_test_best, y_pred_grid_eec),
    'F2_Score': fbeta_score(y_test_best, y_pred_grid_eec, beta=2),
    'ROC_AUC': roc_auc_score(y_test_best, y_proba_grid_eec),
    'PR_AUC': average_precision_score(y_test_best, y_proba_grid_eec),
    'MCC': matthews_corrcoef(y_test_best, y_pred_grid_eec)
}

print(f"\nPerformance Metrics:")
print(f"   Accuracy:  {results_grid_eec['Accuracy']:.4f}")
print(f"   Precision: {results_grid_eec['Precision']:.4f}")
print(f"   Recall:    {results_grid_eec['Recall']:.4f}")
print(f"   F1-Score:  {results_grid_eec['F1_Score']:.4f}")
print(f"   F2-Score:  {results_grid_eec['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_grid_eec['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_grid_eec['PR_AUC']:.4f}")
print(f"   MCC:       {results_grid_eec['MCC']:.4f}")

print(f"\nConfusion Matrix:")
cm_grid_eec = confusion_matrix(y_test_best, y_pred_grid_eec)
print(cm_grid_eec)
print(f"\n   True Negatives:  {cm_grid_eec[0, 0]:,}")
print(f"   False Positives: {cm_grid_eec[0, 1]:,}")
print(f"   False Negatives: {cm_grid_eec[1, 0]:,}")
print(f"   True Positives:  {cm_grid_eec[1, 1]:,}")

print(f"\n📊 Comparison with baseline:")
print(f"   PR-AUC:    {results_eec_10['PR_AUC']:.4f} → {results_grid_eec['PR_AUC']:.4f} ({results_grid_eec['PR_AUC'] - results_eec_10['PR_AUC']:+.4f})")
print(f"   F2-Score:  {results_eec_10['F2_Score']:.4f} → {results_grid_eec['F2_Score']:.4f} ({results_grid_eec['F2_Score'] - results_eec_10['F2_Score']:+.4f})")
print(f"   Recall:    {results_eec_10['Recall']:.4f} → {results_grid_eec['Recall']:.4f} ({results_grid_eec['Recall'] - results_eec_10['Recall']:+.4f})")
print(f"   MCC:       {results_eec_10['MCC']:.4f} → {results_grid_eec['MCC']:.4f} ({results_grid_eec['MCC'] - results_eec_10['MCC']:+.4f})")


In [ ]:
# ==========================================
# OPTION 2: OPTUNA (BAYESIAN OPTIMIZATION)
# ==========================================
print("\n" + "="*80)
print("OPTION 2: OPTUNA - Easy Ensemble + CatBoost (NO FEATURE SELECTION)")
print("="*80)

import optuna
from sklearn.model_selection import cross_val_score

# Suppress Optuna's verbose output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_eec(trial):
    """Objective function for Optuna optimization"""
    
    # Suggest hyperparameters
    params = {
        'iterations': trial.suggest_int('iterations', 50, 250),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 10),
        'random_state': 30,
        'verbose': False
    }
    
    # Create model
    model = EasyEnsembleClassifier(
        estimator=CatBoostClassifier(**params),
        n_estimators=10,  # ✅ Using n=10 for efficiency
        random_state=30,
        n_jobs=-1
    )
    
    # Cross-validation
    scores = cross_val_score(
        model, 
        X_train_best, 
        y_train_best, 
        cv=3, 
        scoring='average_precision',  # PR-AUC
        n_jobs=1
    )
    
    return scores.mean()

print("\n📋 Optuna hyperparameter search space:")
print("   iterations: [50, 250]")
print("   learning_rate: [0.01, 0.2]")
print("   depth: [4, 10]")
print("   l2_leaf_reg: [1, 10]")

print(f"\n🚀 Starting Optuna optimization...")
print(f"   Optimization metric: PR-AUC (average_precision)")
print(f"   Number of trials: 30")
print(f"   Cross-validation: 3-fold")
print(f"   Estimated time: ~60-90 minutes...")

start_time = time.time()

# Create study
study_eec = optuna.create_study(
    direction='maximize',
    study_name='easyensemble_catboost_no_fs'
)

# Optimize
study_eec.optimize(
    objective_eec, 
    n_trials=30,
    show_progress_bar=True
)

optuna_time_eec = time.time() - start_time

print(f"\n✅ Optuna optimization completed in {optuna_time_eec:.2f}s ({optuna_time_eec/60:.1f} minutes)")


In [ ]:
# ==========================================
# OPTUNA RESULTS
# ==========================================
print(f"\n" + "="*80)
print("OPTUNA - BEST PARAMETERS")
print("="*80)

best_params_optuna = study_eec.best_params
print("\n🏆 Best parameters found:")
for param, value in best_params_optuna.items():
    print(f"   {param}: {value}")

print(f"\n📊 Optimization results:")
print(f"   Best CV PR-AUC: {study_eec.best_value:.4f}")
print(f"   Baseline PR-AUC: {results_eec_10['PR_AUC']:.4f}")
print(f"   Improvement: {study_eec.best_value - results_eec_10['PR_AUC']:.4f} ({((study_eec.best_value / results_eec_10['PR_AUC']) - 1) * 100:+.2f}%)")
print(f"   Number of trials: {len(study_eec.trials)}")

# Train final model with best parameters
print(f"\n🔧 Training final model with best Optuna parameters...")
start_time = time.time()

final_model_optuna = EasyEnsembleClassifier(
    estimator=CatBoostClassifier(
        **best_params_optuna,
        random_state=30,
        verbose=False
    ),
    n_estimators=10,
    random_state=30,
    n_jobs=-1
)

final_model_optuna.fit(X_train_best, y_train_best)
final_train_time = time.time() - start_time

# Evaluate on test set
y_pred_optuna_eec = final_model_optuna.predict(X_test_best)
y_proba_optuna_eec = final_model_optuna.predict_proba(X_test_best)[:, 1]

print(f"✅ Final model trained in {final_train_time:.2f}s")

print(f"\n" + "="*80)
print("OPTUNA - TEST SET RESULTS")
print("="*80)

results_optuna_eec = {
    'Method': 'Easy Ensemble (n=10) + Optuna (NO FS)',
    'N_Estimators': 10,
    'N_Features': X_train_best.shape[1],
    'Training_Time': optuna_time_eec,
    'Best_Params': best_params_optuna,
    'CV_PR_AUC': study_eec.best_value,
    'Accuracy': accuracy_score(y_test_best, y_pred_optuna_eec),
    'Precision': precision_score(y_test_best, y_pred_optuna_eec),
    'Recall': recall_score(y_test_best, y_pred_optuna_eec),
    'F1_Score': f1_score(y_test_best, y_pred_optuna_eec),
    'F2_Score': fbeta_score(y_test_best, y_pred_optuna_eec, beta=2),
    'ROC_AUC': roc_auc_score(y_test_best, y_proba_optuna_eec),
    'PR_AUC': average_precision_score(y_test_best, y_proba_optuna_eec),
    'MCC': matthews_corrcoef(y_test_best, y_pred_optuna_eec)
}

print(f"\nPerformance Metrics:")
print(f"   Accuracy:  {results_optuna_eec['Accuracy']:.4f}")
print(f"   Precision: {results_optuna_eec['Precision']:.4f}")
print(f"   Recall:    {results_optuna_eec['Recall']:.4f}")
print(f"   F1-Score:  {results_optuna_eec['F1_Score']:.4f}")
print(f"   F2-Score:  {results_optuna_eec['F2_Score']:.4f}")
print(f"   ROC-AUC:   {results_optuna_eec['ROC_AUC']:.4f}")
print(f"   PR-AUC:    {results_optuna_eec['PR_AUC']:.4f}")
print(f"   MCC:       {results_optuna_eec['MCC']:.4f}")

print(f"\nConfusion Matrix:")
cm_optuna_eec = confusion_matrix(y_test_best, y_pred_optuna_eec)
print(cm_optuna_eec)
print(f"\n   True Negatives:  {cm_optuna_eec[0, 0]:,}")
print(f"   False Positives: {cm_optuna_eec[0, 1]:,}")
print(f"   False Negatives: {cm_optuna_eec[1, 0]:,}")
print(f"   True Positives:  {cm_optuna_eec[1, 1]:,}")

print(f"\n📊 Comparison with baseline:")
print(f"   PR-AUC:    {results_eec_10['PR_AUC']:.4f} → {results_optuna_eec['PR_AUC']:.4f} ({results_optuna_eec['PR_AUC'] - results_eec_10['PR_AUC']:+.4f})")
print(f"   F2-Score:  {results_eec_10['F2_Score']:.4f} → {results_optuna_eec['F2_Score']:.4f} ({results_optuna_eec['F2_Score'] - results_eec_10['F2_Score']:+.4f})")
print(f"   Recall:    {results_eec_10['Recall']:.4f} → {results_optuna_eec['Recall']:.4f} ({results_optuna_eec['Recall'] - results_eec_10['Recall']:+.4f})")
print(f"   MCC:       {results_eec_10['MCC']:.4f} → {results_optuna_eec['MCC']:.4f} ({results_optuna_eec['MCC'] - results_eec_10['MCC']:+.4f})")


In [ ]:
# ==========================================
# COMPARE GRID SEARCH VS OPTUNA
# ==========================================
print("\n" + "="*80)
print("COMPARISON: GRID SEARCH vs OPTUNA vs BASELINE (NO FEATURE SELECTION)")
print("="*80)

tuning_comparison = pd.DataFrame([
    {
        'Method': 'Baseline (no tuning)',
        'N_Estimators': 10,
        'N_Features': X_train.shape[1],  # ✅ FIXED: Use X_train.shape[1] instead
        'Training_Time': results_eec_10['Training_Time'],
        'PR_AUC': results_eec_10['PR_AUC'],
        'F2_Score': results_eec_10['F2_Score'],
        'Recall': results_eec_10['Recall'],
        'MCC': results_eec_10['MCC']
    },
    {
        'Method': 'Grid Search',
        'N_Estimators': 10,
        'N_Features': results_grid_eec['N_Features'],
        'Training_Time': results_grid_eec['Training_Time'],
        'PR_AUC': results_grid_eec['PR_AUC'],
        'F2_Score': results_grid_eec['F2_Score'],
        'Recall': results_grid_eec['Recall'],
        'MCC': results_grid_eec['MCC']
    },
    {
        'Method': 'Optuna',
        'N_Estimators': 10,
        'N_Features': results_optuna_eec['N_Features'],
        'Training_Time': results_optuna_eec['Training_Time'],
        'PR_AUC': results_optuna_eec['PR_AUC'],
        'F2_Score': results_optuna_eec['F2_Score'],
        'Recall': results_optuna_eec['Recall'],
        'MCC': results_optuna_eec['MCC']
    }
])

print("\n📊 Results Comparison:")
display_tuning = tuning_comparison.copy()
for col in ['Training_Time', 'PR_AUC', 'F2_Score', 'Recall', 'MCC']:
    if col == 'Training_Time':
        display_tuning[col] = display_tuning[col].apply(lambda x: f"{x:.2f}s" if x < 120 else f"{x/60:.1f}min")
    else:
        display_tuning[col] = display_tuning[col].apply(lambda x: f"{x:.4f}")

print(display_tuning.to_string(index=False))

# Find best method
best_method_idx = tuning_comparison['PR_AUC'].idxmax()
best_method_name = tuning_comparison.loc[best_method_idx, 'Method']

print(f"\n🏆 WINNER: {best_method_name}")
print(f"   Best PR-AUC: {tuning_comparison.loc[best_method_idx, 'PR_AUC']:.4f}")

print("\n" + "="*80)
print("✅ HYPERPARAMETER TUNING COMPLETE!")
print("="*80)

print("\n💡 Key Findings:")
print(f"   1. Grid Search found PR-AUC: {results_grid_eec['PR_AUC']:.4f}")
print(f"   2. Optuna found PR-AUC: {results_optuna_eec['PR_AUC']:.4f}")
print(f"   3. Baseline PR-AUC: {results_eec_10['PR_AUC']:.4f}")
print(f"\n   Best improvement: {max(results_grid_eec['PR_AUC'], results_optuna_eec['PR_AUC']) - results_eec_10['PR_AUC']:+.4f}")

Hyperarameter tuning

Baseline --> recall 0.031
missed 97% of true positive
the model predicts almost 0

Precision appears perfect only because it predicted 1 for a very tiny number of cases.
The baseline model is unacceptable for business use because it nearly fails to identify the positive class (credit applicants).

SMOTE
Recall and Precision are low, and ROC-AUC is the worst among all methods.

Easy Ensemble
Captures the most positive samples (highest Recall).
Good when the goal is “catch as many positives as possible”.

Very low precision:
The bank would send marketing offers to many non-applicants.
Operational costs increase significantly.

Class Weights 
Best overall balance between Recall and Precision.
Identifies over half of potential applicants 
Lower false-positive rate than Easy Ensemble.

Hyperarameter tuning

In [ ]:
# ==========================================
# SAVE BEST MODEL - OPTUNA WINNER
# ==========================================
from joblib import dump
import pickle
import os

# Create models folder
os.makedirs('models', exist_ok=True)

print("\n" + "="*80)
print("SAVING FINAL MODEL - OPTUNA (Easy Ensemble + CatBoost - NO FEATURE SELECTION)")
print("="*80)

# Save the Optuna model (Easy Ensemble with best hyperparameters)
dump(final_model_optuna, "models/universal_plus_final_model.pkl")
print("✅ Saved: models/universal_plus_final_model.pkl")

# Save ALL features list (NO feature selection - all 199 features)
with open("models/all_features.pkl", 'wb') as f:
    pickle.dump(list(X_train.columns), f)
print("✅ Saved: models/all_features.pkl")

# Save best hyperparameters for documentation
with open("models/best_hyperparameters.pkl", 'wb') as f:
    pickle.dump(best_params_optuna, f)
print("✅ Saved: models/best_hyperparameters.pkl")

print("\n📦 Final model files ready for deployment:")
print("   1. models/universal_plus_final_model.pkl (Easy Ensemble model)")
print("   2. models/all_features.pkl (ALL 199 features)")
print("   3. models/best_hyperparameters.pkl (optimal hyperparameters)")

print("\n🏆 FINAL MODEL SPECIFICATIONS:")
print("   Method: Easy Ensemble + CatBoost + Optuna (NO Feature Selection)")
print("   n_estimators: 10")
print(f"   iterations: {best_params_optuna['iterations']}")
print(f"   learning_rate: {best_params_optuna['learning_rate']:.4f}")
print(f"   depth: {best_params_optuna['depth']}")
print(f"   l2_leaf_reg: {best_params_optuna['l2_leaf_reg']}")
print("   Features: 199 (all features)")
print(f"   PR-AUC: {results_optuna_eec['PR_AUC']:.4f}")
print(f"   Recall: {results_optuna_eec['Recall']:.4f}")
print(f"   F2-Score: {results_optuna_eec['F2_Score']:.4f}")
print(f"   MCC: {results_optuna_eec['MCC']:.4f}")

# Print confusion matrix info
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives:  {cm_optuna_eec[0, 0]:,}")
print(f"   False Positives: {cm_optuna_eec[0, 1]:,}")
print(f"   False Negatives: {cm_optuna_eec[1, 0]:,}")
print(f"   True Positives:  {cm_optuna_eec[1, 1]:,}")

print("\n" + "="*80)
print("✅ FINAL MODEL SAVED SUCCESSFULLY!")
print("="*80)

In [ ]:
# ==========================================
# LOAD MODEL FOR INFERENCE
# ==========================================
from joblib import load
import pickle
import pandas as pd

print("\n" + "="*80)
print("LOADING FINAL MODEL FOR PREDICTIONS")
print("="*80)

# Load the trained Easy Ensemble model
final_model = load("models/universal_plus_final_model.pkl")
print("✅ Loaded model: Easy Ensemble + CatBoost (Optuna)")

# Load all features list
with open("models/all_features.pkl", 'rb') as f:
    all_features = pickle.load(f)
print(f"✅ Loaded {len(all_features)} features (all features - no selection)")

# Load best hyperparameters (optional - for documentation)
with open("models/best_hyperparameters.pkl", 'rb') as f:
    best_params = pickle.load(f)
print(f"✅ Loaded best hyperparameters")

print("\n📋 Model Configuration:")
for param, value in best_params.items():
    print(f"   {param}: {value}")

# ==========================================
# MAKE PREDICTIONS ON TEST SET
# ==========================================
print("\n" + "="*80)
print("MAKING PREDICTIONS ON TEST SET")
print("="*80)

# Use ALL features (no selection)
X_test_final = X_test[all_features]
print(f"✅ Test data shape: {X_test_final.shape}")

# Make predictions
y_pred_final = final_model.predict(X_test_final)
y_proba_final = final_model.predict_proba(X_test_final)[:, 1]

print(f"✅ Predictions generated:")
print(f"   Binary predictions: {y_pred_final.shape}")
print(f"   Probability scores: {y_proba_final.shape}")

# Quick performance check
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import fbeta_score, roc_auc_score, average_precision_score, matthews_corrcoef

print("\n📊 FINAL MODEL PERFORMANCE:")
print(f"   Accuracy:  {accuracy_score(y_test, y_pred_final):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"   Recall:    {recall_score(y_test, y_pred_final):.4f}")
print(f"   F1-Score:  {f1_score(y_test, y_pred_final):.4f}")
print(f"   F2-Score:  {fbeta_score(y_test, y_pred_final, beta=2):.4f}")
print(f"   ROC-AUC:   {roc_auc_score(y_test, y_proba_final):.4f}")
print(f"   PR-AUC:    {average_precision_score(y_test, y_proba_final):.4f}")
print(f"   MCC:       {matthews_corrcoef(y_test, y_pred_final):.4f}")

print("\n✅ Model ready for production deployment!")